# SQD Demo — N₂ / 6-31G (active-space benchmark)

This notebook runs SQD on N₂ in the 6-31G basis with the two 1s core orbitals
frozen, matching the active space used in the
[qiskit-addon-sqd tutorial](https://qiskit.github.io/qiskit-addon-sqd/tutorials/01_chemistry_hamiltonian.html).
It is a more demanding sampling test than the STO-3G notebook and a good
reference point before scaling up to fragment-method calculations.

## Overview

The SQD algorithm combines:
1. **Quantum sampling** — LUCJ ansatz built by `ffsim.qiskit.lucj_pass_manager`,
   submitted to IBM Quantum via QRMI
2. **Classical post-processing** — configuration recovery + SBD diagonalization
   via `qiskit-addon-sqd`

## Credentials

Credentials are read from **environment variables**, never from config files.

1. Copy the example env file once:
   ```zsh
   cp examples/notebook_demos/sqd_demos/.env.example \
      examples/notebook_demos/sqd_demos/.env
   ```
2. Fill in `.env` with your IBM Quantum IAM API key and CRN:
   ```
   ibm_kingston_QRMI_IBM_QCS_IAM_APIKEY=<your-iam-api-key>
   ibm_kingston_QRMI_IBM_QCS_SERVICE_CRN=<your-crn>
   ```
   The prefix must be **lowercase** and match `QRMI_JOB_QPU_RESOURCES` exactly.

The `.env` file is gitignored — it will never be committed.

> **HPC / Slurm:** Skip the `.env` file. The Slurm SPANK plugin injects all
> `QRMI_*` variables automatically when the job requests a QPU resource.

## Setup

In [9]:
import os
import yaml
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from pyscf import gto, scf, cc, mcscf, ao2mo

from quantum_fragment_methods.qpu import QRMIBackend
from quantum_fragment_methods.application.solvers.quantum_zoo.sqd import SQDSolver

DEMO_DIR = Path('__file__').resolve().parent

## 1. Load Configuration

In [10]:
config_path = DEMO_DIR / 'config_N2_6-31G_demo.yaml'

with open(config_path) as f:
    config = yaml.safe_load(f)

qpu_config = config['qpu']
sqd_config = config['sqd']

print(f'Config  : {config_path}')
print(f'Backend : {qpu_config["backend_name"]}')
print(f'Shots   : {qpu_config["sampler_options"]["default_shots"]}')
print(f'SQD iter: {sqd_config["iterations"]}  batches: {sqd_config["n_batches"]}  samples: {sqd_config["samples_per_batch"]}')

Config  : /Users/thaddeuspellegrini/Code/QuantumAlgorithmEngineering/QDC_2026/quantum-fragment-methods/examples/notebook_demos/sqd_demos/config_N2_6-31G_demo.yaml
Backend : ibm_kingston
Shots   : 100000
SQD iter: 5  batches: 1  samples: 1000


## 2. Define Molecular System — N₂ / 6-31G

In [11]:
mol = gto.Mole()
mol.atom = '''
N 0.0 0.0 0.0
N 1.0 0.0 0.0
'''
mol.basis = '6-31G'
mol.build()

print(f'Orbitals : {mol.nao}')
print(f'Electrons: {mol.nelectron}')

Orbitals : 18
Electrons: 14


## 3. Hartree-Fock + Active-Space Hamiltonian

The two 1s core orbitals are frozen to reduce cost while matching the
active space used in the qiskit-addon-sqd tutorial.

In [12]:
mf = scf.RHF(mol)
mf.kernel()
print(f'HF energy: {mf.e_tot:.8f}')

# Active space: freeze the two 1s core orbitals
n_frozen     = 2
active_space = range(n_frozen, mol.nao_nr())
norb         = len(active_space)
nelec_total  = int(sum(mf.mo_occ[active_space]))
num_elec_a   = (nelec_total + mol.spin) // 2
num_elec_b   = (nelec_total - mol.spin) // 2
nelec        = (num_elec_a, num_elec_b)

cas        = mcscf.CASCI(mf, norb, nelec)
mo         = cas.sort_mo(active_space, base=0)
h1e, nuc_energy = cas.get_h1cas(mo)
h2e        = ao2mo.restore(1, cas.get_h2cas(mo), norb)

print(f'Active space: norb={norb}  nelec={nelec}  nuc={nuc_energy:.8f}')

converged SCF energy = -108.835236570775
HF energy: -108.83523657
Active space: norb=16  nelec=(5, 5)  nuc=-76.23110254


## 4. CCSD Amplitudes for LUCJ Initialization

In [13]:
# Frozen-core CCSD to match the active space
ccsd = cc.CCSD(mf, frozen=n_frozen).run()
t1, t2 = ccsd.t1, ccsd.t2
print(f't1 shape: {t1.shape}  t2 shape: {t2.shape}')

E(CCSD) = -109.0398256929948  E_corr = -0.2045891222202483
t1 shape: (5, 11)  t2 shape: (5, 5, 11, 11)


## 5. Initialize QRMI Backend

Credentials are loaded from `.env` (or already set by Slurm). The backend
name is taken from `QRMI_JOB_QPU_RESOURCES` if set, otherwise falls back
to the value in the config file.

In [14]:
# Load .env without overriding variables already injected by Slurm SPANK plugin
load_dotenv(DEMO_DIR / '.env', override=False)

BACKEND_NAME = os.environ.get('QRMI_JOB_QPU_RESOURCES', qpu_config['backend_name'])
os.environ.setdefault('QRMI_JOB_QPU_RESOURCES', BACKEND_NAME)
os.environ.setdefault('QRMI_JOB_QPU_TYPES', 'ibm-quantum-compute-service')

backend = QRMIBackend({'backend_name': BACKEND_NAME})
backend.initialize()
backend.get_backend()

props = backend.get_backend_properties()
print(f'Backend: {props["backend_name"]}  ({props["resource_type"]})')

Backend: ibm_kingston  (ResourceType.IBMQuantumComputeService)


## 6. Run SQD

- Builds a LUCJ circuit via `ffsim.qiskit.lucj_pass_manager` using the
  pre-computed t1/t2 amplitudes
- Submits to IBM Quantum via QRMI
- Checkpoints to `sqd_results/` (re-running resumes from `counts.npy` if present)
- Runs SBD post-processing

In [15]:
solver       = SQDSolver(backend, config=sqd_config)
workflow_dir = DEMO_DIR / 'sqd_results' / 'N2_6-31G'

result = solver.solve(
    h1e=h1e,
    h2e=h2e,
    norb=norb,
    nelec=nelec,
    t1=t1,
    t2=t2,
    workflow_path=str(workflow_dir),
    wait_for_completion=True,
    force_resubmit=False,          # set True to clear checkpoint and resubmit
)

print(f'Electronic energy: {result.energy:.8f} Ha')
print(f'Total energy     : {result.energy + nuc_energy:.8f} Ha')

  QPU job submitted: dai5hsphvn6c73csqe10
  Backend: ibm_kingston  checkpoint: /Users/thaddeuspellegrini/Code/QuantumAlgorithmEngineering/QDC_2026/quantum-fragment-methods/examples/notebook_demos/sqd_demos/sqd_results/N2_6-31G/job_id.txt
  Polling job dai5hsphvn6c73csqe10 every 30s (max 300s) ...
  00s] dai5hsphvn6c73csqe10 — RUNNING
  30s] dai5hsphvn6c73csqe10 — RUNNING
  00s] dai5hsphvn6c73csqe10 — RUNNING
  30s] dai5hsphvn6c73csqe10 — RUNNING
  01s] dai5hsphvn6c73csqe10 — RUNNING
  31s] dai5hsphvn6c73csqe10 — DONE
  ✓ Job completed (2m31s)
  SQD: 5 iter × 1 batches × 1000 samples  backend=python
  SBD complete — E = -32.81412749 Ha
Electronic energy: -32.81412749 Ha
Total energy     : -109.04523003 Ha


## 7. CASCI Comparison

Reference value from the qiskit-addon-sqd tutorial
(CASCI for the same frozen-core active space).

In [16]:
CASCI_energy = -109.04667800  # from qiskit-addon-sqd tutorial

sqd_total = result.energy + nuc_energy
sqd_error = abs(sqd_total - CASCI_energy)

print(f'HF    energy : {mf.e_tot:.8f} Ha')
print(f'CASCI energy : {CASCI_energy:.8f} Ha  (reference)')
print(f'SQD   energy : {sqd_total:.8f} Ha')
print(f'SQD vs CASCI : {sqd_error:.8f} Ha  ({sqd_error * 627.5:.4f} kcal/mol)')

HF    energy : -108.83523657 Ha
CASCI energy : -109.04667800 Ha  (reference)
SQD   energy : -109.04523003 Ha
SQD vs CASCI : 0.00144797 Ha  (0.9086 kcal/mol)
